# ResNet-18 – Transfer Learning trên CIFAR-10

Notebook thực hiện huấn luyện và so sánh hai chiến lược Transfer Learning sử dụng kiến trúc ResNet-18 trên bộ dữ liệu CIFAR-10.

Hai mô hình được sử dụng:

- **Model A – FC Only:** đóng băng backbone và chỉ huấn luyện lớp Fully Connected cuối.
- **Model B – Layer4 + FC:** fine-tune Layer4 và lớp Fully Connected.

Mục tiêu là so sánh số lượng tham số huấn luyện, Loss, Accuracy và thời gian huấn luyện của hai chiến lược.

## 1. Import thư viện và thiết lập project

Phần này import các thư viện cần thiết, xác định thư mục gốc của project và lựa chọn thiết bị tính toán CPU/GPU.

In [4]:
# ============================================================
# CELL 2 - IMPORT + PROJECT_ROOT
# ============================================================

import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn


# ============================================================
# 1. TÌM PROJECT_ROOT
# ============================================================

def find_project_root():
    """
    Tự tìm thư mục Practice_2 khi chạy trên:
    - VS Code / Windows
    - Google Colab
    """

    current = Path.cwd().resolve()

    # Kiểm tra thư mục hiện tại và các thư mục cha
    for path in [current, *current.parents]:

        # Trường hợp đang đứng ngay trong Practice_2
        if path.name == "Practice_2" and (path / "src").exists():
            return path

        # Trường hợp Practice_2 nằm bên trong thư mục hiện tại
        candidate = path / "Practice_2"

        if candidate.exists() and (candidate / "src").exists():
            return candidate.resolve()

    # Đường dẫn chuẩn khi chạy trên Google Colab
    colab_path = Path(
        "/content/UTH-Deep-Learning-nhom2/Practice_2"
    )

    if colab_path.exists() and (colab_path / "src").exists():
        return colab_path.resolve()

    raise FileNotFoundError(
        f"Không tìm thấy thư mục Practice_2.\n"
        f"Current directory: {current}"
    )


PROJECT_ROOT = find_project_root()


# ============================================================
# 2. THÊM PROJECT VÀO PYTHON PATH
# ============================================================

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# 3. IMPORT CODE DÙNG CHUNG CỦA NHÓM
# ============================================================

from src.data import (
    DataConfig,
    build_dataloaders,
)

from src.trainer import (
    get_device,
    train_model,
)

from src.models.resnet18 import (
    build_resnet18_fc_only,
    build_resnet18_layer4_fc,
)


# ============================================================
# 4. HÀM ĐẾM PARAMETERS
# ============================================================

def count_total_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
    )


def count_trainable_parameters(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def count_frozen_parameters(model):
    return (
        count_total_parameters(model)
        - count_trainable_parameters(model)
    )


# ============================================================
# 5. DEVICE
# ============================================================

device = get_device()


# ============================================================
# 6. KIỂM TRA PROJECT
# ============================================================

print("===== PROJECT SETUP =====")
print("Current dir  :", Path.cwd())
print("PROJECT_ROOT :", PROJECT_ROOT)
print("PyTorch      :", torch.__version__)
print("CUDA         :", torch.cuda.is_available())
print("Device       :", device)

if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))

===== PROJECT SETUP =====
Current dir  : d:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\notebooks
PROJECT_ROOT : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2
PyTorch      : 2.13.0+cpu
CUDA         : False
Device       : cpu


## 2. Chuẩn bị dữ liệu CIFAR-10

Xây dựng DataLoader cho ba tập dữ liệu:

- Training set
- Validation set
- Test set

Dữ liệu CIFAR-10 gồm 10 lớp ảnh và được tiền xử lý trước khi đưa vào ResNet-18.

In [6]:
# ============================================================
# CELL 3 - DATALOADER CHUNG
# ============================================================

data_config = DataConfig(
    data_dir=PROJECT_ROOT / "data" / "raw",
    batch_size=32,
    num_workers=0,
)

data_bundle = build_dataloaders(data_config)

train_loader = data_bundle["train_loader"]
val_loader = data_bundle["val_loader"]
test_loader = data_bundle["test_loader"]
class_names = data_bundle["class_names"]

print("===== DATALOADER READY =====")
print("Data directory :", data_config.data_dir)

print("Train samples  :", len(train_loader.dataset))
print("Val samples    :", len(val_loader.dataset))
print("Test samples   :", len(test_loader.dataset))

print("Train batches  :", len(train_loader))
print("Val batches    :", len(val_loader))
print("Test batches   :", len(test_loader))

print("Classes        :", class_names)
print("Number classes :", len(class_names))

===== DATALOADER READY =====
Data directory : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\data\raw
Train samples  : 45000
Val samples    : 5000
Test samples   : 10000
Train batches  : 1407
Val batches    : 157
Test batches   : 313
Classes        : ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Number classes : 10


## 3. Xây dựng hai chiến lược ResNet-18

Hai mô hình ResNet-18 được xây dựng để so sánh hai chiến lược Transfer Learning:

### Model A – FC Only
Đóng băng các tầng backbone và chỉ huấn luyện lớp Fully Connected cuối.

### Model B – Layer4 + FC
Fine-tune Layer4 và lớp Fully Connected, trong khi các tầng trước vẫn được đóng băng.

In [10]:
# ============================================================
# CELL 4 - XÂY RESNET-18 A VÀ B
# ============================================================

num_classes = len(class_names)

# ResNet-18 A:
# Chỉ huấn luyện lớp FC
model_a = build_resnet18_fc_only(
    num_classes=num_classes
)

# ResNet-18 B:
# Fine-tuning layer4 + FC
model_b = build_resnet18_layer4_fc(
    num_classes=num_classes
)

# Đưa model về đúng device hiện tại
model_a = model_a.to(device)
model_b = model_b.to(device)

print("===== MODELS READY =====")
print("Model A : ResNet-18 - FC only")
print("Model B : ResNet-18 - Layer4 + FC")
print("Classes :", num_classes)
print("Device  :", device)

===== MODELS READY =====
Model A : ResNet-18 - FC only
Model B : ResNet-18 - Layer4 + FC
Classes : 10
Device  : cpu


## 4. So sánh số lượng tham số

Thống kê:

- Total Parameters
- Trainable Parameters
- Frozen Parameters

Mục đích là đánh giá mức độ khác nhau về chi phí huấn luyện giữa Model A và Model B.

In [11]:
# ============================================================
# CELL 5 - SO SÁNH TRAINABLE PARAMETERS
# ============================================================

total_params_a = count_total_parameters(model_a)
trainable_params_a = count_trainable_parameters(model_a)
frozen_params_a = count_frozen_parameters(model_a)

total_params_b = count_total_parameters(model_b)
trainable_params_b = count_trainable_parameters(model_b)
frozen_params_b = count_frozen_parameters(model_b)

print("===== PARAMETER COMPARISON =====")

print("\nResNet-18 A - FC only")
print("Total params     :", f"{total_params_a:,}")
print("Trainable params :", f"{trainable_params_a:,}")
print("Frozen params    :", f"{frozen_params_a:,}")

print("\nResNet-18 B - Layer4 + FC")
print("Total params     :", f"{total_params_b:,}")
print("Trainable params :", f"{trainable_params_b:,}")
print("Frozen params    :", f"{frozen_params_b:,}")

===== PARAMETER COMPARISON =====

ResNet-18 A - FC only
Total params     : 11,181,642
Trainable params : 5,130
Frozen params    : 11,176,512

ResNet-18 B - Layer4 + FC
Total params     : 11,181,642
Trainable params : 8,398,858
Frozen params    : 2,782,784


## 5. Kiểm tra Forward Pass

Thực hiện một lượt truyền xuôi trên một batch dữ liệu để kiểm tra:

- Kích thước dữ liệu đầu vào
- Kích thước nhãn
- Kích thước đầu ra của Model A
- Kích thước đầu ra của Model B

Đầu ra mong đợi của mỗi mô hình có 10 giá trị tương ứng với 10 lớp của CIFAR-10.

In [12]:
# ============================================================
# CELL 6 - FORWARD PASS CHECK
# ============================================================

# Lấy 1 batch từ tập train
images, labels = next(iter(train_loader))

# Đưa dữ liệu lên cùng device với model
images = images.to(device)
labels = labels.to(device)

# Chuyển sang eval để kiểm tra forward pass
model_a.eval()
model_b.eval()

# Không tính gradient vì chỉ kiểm tra
with torch.no_grad():
    outputs_a = model_a(images)
    outputs_b = model_b(images)

print("===== FORWARD PASS CHECK =====")
print("Device         :", device)
print("Images device  :", images.device)
print("Labels device  :", labels.device)

print("\nInput shape    :", images.shape)
print("Labels shape   :", labels.shape)
print("Output A shape :", outputs_a.shape)
print("Output B shape :", outputs_b.shape)

===== FORWARD PASS CHECK =====
Device         : cpu
Images device  : cpu
Labels device  : cpu

Input shape    : torch.Size([32, 3, 224, 224])
Labels shape   : torch.Size([32])
Output A shape : torch.Size([32, 10])
Output B shape : torch.Size([32, 10])


## 6. Cấu hình huấn luyện

Thiết lập các tham số dùng cho quá trình training:

- Số epoch
- Learning rate
- Batch size
- Loss function
- Optimizer
- Checkpoint

Cả hai mô hình sử dụng cùng cấu hình cơ bản để việc so sánh được nhất quán.

In [13]:
# ============================================================
# CELL 7 - CẤU HÌNH TRAINING CHUNG
# ============================================================

# Số epoch
NUM_EPOCHS = 5

# Learning rate
LEARNING_RATE = 0.001

# Loss function dùng chung
criterion = nn.CrossEntropyLoss()

# Chỉ đưa các parameter có requires_grad=True vào optimizer
optimizer_a = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_a.parameters()),
    lr=LEARNING_RATE
)

optimizer_b = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_b.parameters()),
    lr=LEARNING_RATE
)

# ------------------------------------------------------------
# Thư mục checkpoint
# ------------------------------------------------------------

checkpoint_dir = PROJECT_ROOT / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_a = checkpoint_dir / "resnet18_fc_only_best.pth"
checkpoint_b = checkpoint_dir / "resnet18_layer4_fc_best.pth"

# ------------------------------------------------------------
# Hiển thị cấu hình
# ------------------------------------------------------------

print("===== TRAINING CONFIG =====")
print("Epochs           :", NUM_EPOCHS)
print("Learning rate    :", LEARNING_RATE)
print("Batch size       :", data_config.batch_size)
print("Loss function    : CrossEntropyLoss")
print("Optimizer A      : Adam")
print("Optimizer B      : Adam")
print("Device           :", device)

print("\nCheckpoint A     :", checkpoint_a)
print("Checkpoint B     :", checkpoint_b)

===== TRAINING CONFIG =====
Epochs           : 5
Learning rate    : 0.001
Batch size       : 32
Loss function    : CrossEntropyLoss
Optimizer A      : Adam
Optimizer B      : Adam
Device           : cpu

Checkpoint A     : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\checkpoints\resnet18_fc_only_best.pth
Checkpoint B     : D:\DEEP\UTH-Deep-Learning-nhom2\Practice_2\checkpoints\resnet18_layer4_fc_best.pth


## 7. Huấn luyện Model A – FC Only

Huấn luyện ResNet-18 theo chiến lược chỉ cập nhật lớp Fully Connected cuối.

Trong quá trình training, các giá trị Train Loss, Validation Loss, Train Accuracy và Validation Accuracy được lưu lại để phục vụ việc đánh giá và trực quan hóa kết quả.

In [ ]:
# ============================================================
# CELL 8 - TRAIN RESNET-18 A
# ============================================================

log_dir_a = PROJECT_ROOT / "runs" / "resnet18_fc_only"

print("===== TRAIN RESNET-18 A =====")
print("Strategy         : FC only")
print("Trainable params :", f"{trainable_params_a:,}")
print("Optimizer        : Adam")
print("Learning rate    :", LEARNING_RATE)
print("Batch size       :", data_config.batch_size)
print("Epochs           :", NUM_EPOCHS)
print("Device           :", device)
print("Checkpoint       :", checkpoint_a)

# Đo thời gian training
start_time_a = time.time()

history_a = train_model(
    model=model_a,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_a,
    num_epochs=NUM_EPOCHS,
    log_dir=log_dir_a,
    checkpoint_path=checkpoint_a,
)

training_time_a = time.time() - start_time_a

# Lấy epoch tốt nhất trực tiếp từ history
best_val_acc_a = max(history_a["val_acc"])
best_epoch_a = history_a["val_acc"].index(best_val_acc_a) + 1

print("\n===== RESNET-18 A COMPLETED =====")
print("Training time     :", f"{training_time_a:.2f} seconds")
print("Best Val Accuracy :", f"{best_val_acc_a:.2f}%")
print("Best Epoch        :", f"{best_epoch_a}/{NUM_EPOCHS}")
print("Checkpoint        :", checkpoint_a)

## 8. Huấn luyện Model B – Layer4 + FC

Huấn luyện ResNet-18 theo chiến lược fine-tune Layer4 và lớp Fully Connected.

Kết quả training được lưu lại để so sánh trực tiếp với Model A.

In [ ]:
# ============================================================
# CELL 9 - TRAIN RESNET-18 B
# ============================================================

log_dir_b = PROJECT_ROOT / "runs" / "resnet18_layer4_fc"

print("===== TRAIN RESNET-18 B =====")
print("Strategy         : Layer4 + FC")
print("Trainable params :", f"{trainable_params_b:,}")
print("Optimizer        : Adam")
print("Learning rate    :", LEARNING_RATE)
print("Batch size       :", data_config.batch_size)
print("Epochs           :", NUM_EPOCHS)
print("Device           :", device)
print("Checkpoint       :", checkpoint_b)

# Đo thời gian training
start_time_b = time.time()

history_b = train_model(
    model=model_b,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer_b,
    num_epochs=NUM_EPOCHS,
    log_dir=log_dir_b,
    checkpoint_path=checkpoint_b,
)

training_time_b = time.time() - start_time_b

# Lấy epoch tốt nhất trực tiếp từ history
best_val_acc_b = max(history_b["val_acc"])
best_epoch_b = history_b["val_acc"].index(best_val_acc_b) + 1

print("\n===== RESNET-18 B COMPLETED =====")
print("Training time     :", f"{training_time_b:.2f} seconds")
print("Best Val Accuracy :", f"{best_val_acc_b:.2f}%")
print("Best Epoch        :", f"{best_epoch_b}/{NUM_EPOCHS}")
print("Checkpoint        :", checkpoint_b)

## 9. Tổng hợp và lưu kết quả

Tổng hợp kết quả của hai chiến lược vào bảng so sánh gồm:

- Số lượng tham số
- Best Validation Accuracy
- Best Epoch
- Training Time

Bảng kết quả được lưu thành file `resnet18_summary.csv`.

In [ ]:
# ============================================================
# CELL 10 - BẢNG SO SÁNH + LƯU CSV
# ============================================================

# Thư mục lưu bảng kết quả
table_dir = PROJECT_ROOT / "results" / "tables"
table_dir.mkdir(parents=True, exist_ok=True)

csv_path = table_dir / "resnet18_summary.csv"

# ------------------------------------------------------------
# Tạo bảng từ kết quả training thật
# ------------------------------------------------------------

comparison_df = pd.DataFrame({
    "Model": [
        "ResNet-18 A - FC only",
        "ResNet-18 B - Layer4 + FC",
    ],

    "Strategy": [
        "FC only",
        "Layer4 + FC",
    ],

    "Total Params": [
        total_params_a,
        total_params_b,
    ],

    "Trainable Params": [
        trainable_params_a,
        trainable_params_b,
    ],

    "Frozen Params": [
        frozen_params_a,
        frozen_params_b,
    ],

    "Best Val Accuracy (%)": [
        best_val_acc_a,
        best_val_acc_b,
    ],

    "Best Epoch": [
        best_epoch_a,
        best_epoch_b,
    ],

    "Training Time (s)": [
        training_time_a,
        training_time_b,
    ],
})

# ------------------------------------------------------------
# Làm tròn số cho bảng
# ------------------------------------------------------------

comparison_df["Best Val Accuracy (%)"] = (
    comparison_df["Best Val Accuracy (%)"].round(2)
)

comparison_df["Training Time (s)"] = (
    comparison_df["Training Time (s)"].round(2)
)

# ------------------------------------------------------------
# Hiển thị bảng
# ------------------------------------------------------------

print("===== RESNET-18 COMPARISON =====")

display(comparison_df)

# ------------------------------------------------------------
# Lưu CSV
# ------------------------------------------------------------

comparison_df.to_csv(
    csv_path,
    index=False
)

print("\nCSV saved successfully.")
print("Saved CSV:", csv_path)

## 10. Trực quan hóa Loss

Vẽ đường Train Loss và Validation Loss của Model A và Model B theo từng epoch.

Biểu đồ được lưu thành file `resnet18_loss.png`.

In [ ]:
# ============================================================
# CELL 11 - LOSS CURVES + SAVE PNG
# ============================================================

# Thư mục lưu hình
figure_dir = PROJECT_ROOT / "results" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

loss_path = figure_dir / "resnet18_loss.png"

# Số epoch dựa trực tiếp trên history
epochs_a = range(1, len(history_a["train_loss"]) + 1)
epochs_b = range(1, len(history_b["train_loss"]) + 1)

# ------------------------------------------------------------
# Vẽ biểu đồ Loss
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_a,
    history_a["train_loss"],
    marker="o",
    label="A - Train Loss"
)

plt.plot(
    epochs_a,
    history_a["val_loss"],
    marker="o",
    label="A - Val Loss"
)

plt.plot(
    epochs_b,
    history_b["train_loss"],
    marker="o",
    label="B - Train Loss"
)

plt.plot(
    epochs_b,
    history_b["val_loss"],
    marker="o",
    label="B - Val Loss"
)

plt.title("ResNet-18 A vs B - Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.xticks(range(1, NUM_EPOCHS + 1))
plt.grid(True)
plt.legend()
plt.tight_layout()

# ------------------------------------------------------------
# Tự lưu PNG
# ------------------------------------------------------------

plt.savefig(
    loss_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("===== LOSS FIGURE SAVED =====")
print("Saved PNG:", loss_path)

## 11. Trực quan hóa Accuracy

Vẽ đường Train Accuracy và Validation Accuracy của Model A và Model B theo từng epoch.

Biểu đồ được lưu thành file `resnet18_accuracy.png`.

In [ ]:
# ============================================================
# CELL 12 - ACCURACY CURVES + SAVE PNG
# ============================================================

# Thư mục lưu hình
figure_dir = PROJECT_ROOT / "results" / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)

accuracy_path = figure_dir / "resnet18_accuracy.png"

# Epoch lấy trực tiếp từ history
epochs_a = range(1, len(history_a["train_acc"]) + 1)
epochs_b = range(1, len(history_b["train_acc"]) + 1)

# ------------------------------------------------------------
# Vẽ biểu đồ Accuracy
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    epochs_a,
    history_a["train_acc"],
    marker="o",
    label="A - Train Accuracy"
)

plt.plot(
    epochs_a,
    history_a["val_acc"],
    marker="o",
    label="A - Val Accuracy"
)

plt.plot(
    epochs_b,
    history_b["train_acc"],
    marker="o",
    label="B - Train Accuracy"
)

plt.plot(
    epochs_b,
    history_b["val_acc"],
    marker="o",
    label="B - Val Accuracy"
)

plt.title("ResNet-18 A vs B - Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.xticks(range(1, NUM_EPOCHS + 1))
plt.grid(True)
plt.legend()
plt.tight_layout()

# ------------------------------------------------------------
# Tự lưu PNG
# ------------------------------------------------------------

plt.savefig(
    accuracy_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("===== ACCURACY FIGURE SAVED =====")
print("Saved PNG:", accuracy_path)

## 12. Kiểm tra các file kết quả

Kiểm tra checkpoint của hai mô hình và xác nhận ba file kết quả chính đã được tạo thành công:

- `resnet18_summary.csv`
- `resnet18_loss.png`
- `resnet18_accuracy.png`

In [ ]:
# ============================================================
# CELL 13 - VERIFY CHECKPOINTS + RESULT FILES
# ============================================================

print("===== CHECKPOINT VERIFICATION =====")

print("\nResNet-18 A")
print("Path   :", checkpoint_a)
print("Exists :", checkpoint_a.exists())

print("\nResNet-18 B")
print("Path   :", checkpoint_b)
print("Exists :", checkpoint_b.exists())


# ============================================================
# KIỂM TRA 3 FILE KẾT QUẢ
# ============================================================

csv_path = PROJECT_ROOT / "results" / "tables" / "resnet18_summary.csv"
loss_path = PROJECT_ROOT / "results" / "figures" / "resnet18_loss.png"
accuracy_path = PROJECT_ROOT / "results" / "figures" / "resnet18_accuracy.png"

print("\n===== RESULT FILE VERIFICATION =====")

print("\nCSV")
print("Path   :", csv_path)
print("Exists :", csv_path.exists())

print("\nLoss PNG")
print("Path   :", loss_path)
print("Exists :", loss_path.exists())

print("\nAccuracy PNG")
print("Path   :", accuracy_path)
print("Exists :", accuracy_path.exists())


# ============================================================
# TỔNG KẾT
# ============================================================

all_checkpoints_ok = (
    checkpoint_a.exists()
    and checkpoint_b.exists()
)

all_results_ok = (
    csv_path.exists()
    and loss_path.exists()
    and accuracy_path.exists()
)

print("\n===== FINAL VERIFICATION =====")
print("Checkpoints ready :", all_checkpoints_ok)
print("Result files ready:", all_results_ok)

if all_checkpoints_ok and all_results_ok:
    print("\nALL RESNET-18 OUTPUTS ARE READY.")
else:
    print("\nSome files are still missing.")

# Kết luận

Trong bài thực hành này, nhóm sử dụng **ResNet-18 pretrained trên ImageNet** để thực hiện bài toán phân loại ảnh trên bộ dữ liệu **CIFAR-10** với 10 lớp.

Hai chiến lược Transfer Learning được so sánh:

### ResNet-18 A – FC Only
- Giữ nguyên và đóng băng các tầng convolution của ResNet-18.
- Chỉ huấn luyện lớp Fully Connected (FC) cuối cùng.
- Số lượng tham số cần huấn luyện ít.
- Thời gian huấn luyện thấp hơn.

### ResNet-18 B – Layer4 + FC
- Giữ nguyên các tầng trước của ResNet-18.
- Fine-tune `layer4` và lớp `fc`.
- Có nhiều tham số được huấn luyện hơn Model A.
- Cho phép mô hình thích nghi tốt hơn với đặc trưng của CIFAR-10 nhưng yêu cầu nhiều tài nguyên tính toán hơn.

## Kết quả so sánh

Kết quả thực nghiệm được đánh giá dựa trên:

- Training Loss
- Validation Loss
- Training Accuracy
- Validation Accuracy
- Best Validation Accuracy
- Training Time
- Số lượng Trainable Parameters

Các kết quả được lưu tại:

- `results/tables/resnet18_summary.csv`
- `results/figures/resnet18_loss.png`
- `results/figures/resnet18_accuracy.png`

Qua việc so sánh hai chiến lược, có thể đánh giá sự đánh đổi giữa **chi phí huấn luyện** và **khả năng thích nghi của mô hình** khi áp dụng Transfer Learning với ResNet-18.